In [1]:
!pip install chronos-forecasting
!pip install transformers accelerate

In [1]:
import torch
import pandas as pd
import numpy as np
from chronos import Chronos2Pipeline
from sklearn.metrics import r2_score,mean_squared_error
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv
import os

/mnt/c/Users/RazorVision/Nextcloud/uni/7 semestar/vrnmp/project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
from huggingface_hub import login
login(token=os.getenv("TOKEN"))

In [4]:
# df = pd.read_csv("/content/bitola_final_data.csv")
df = pd.read_csv("../data/training/bitola_final_data.csv")
df.head()

,timestamp,sensorId,city,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,58.333333,28.333333,13.666667,943.0,12.00,8.3,63.645833,...,8.3,6.98,0.000000,1.000000,-0.5,0.866025,-0.433884,-0.900969,0,1
1,2023-12-01 01:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,13.750000,5.000000,943.0,12.00,9.3,64.750000,...,9.3,7.50,0.258819,0.965926,-0.5,0.866025,-0.433884,-0.900969,0,1
2,2023-12-01 02:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,60.000000,10.000000,4.500000,942.5,12.25,8.3,65.312500,...,8.3,6.74,0.500000,0.866025,-0.5,0.866025,-0.433884,-0.900969,0,1
3,2023-12-01 03:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.250000,9.750000,4.500000,942.0,13.00,9.4,64.854167,...,9.4,7.84,0.707107,0.707107,-0.5,0.866025,-0.433884,-0.900969,0,1
4,2023-12-01 04:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,bitola,59.500000,5.000000,2.000000,942.0,13.00,9.0,65.062500,...,9.0,8.32,0.866025,0.500000,-0.5,0.866025,-0.433884,-0.900969,0,1


In [5]:
df.describe()

,humidity,pm10,pm25,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
count,192984.000000,149772.000000,149781.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,...,192984.000000,192984.000000,1.929840e+05,1.929840e+05,192984.000000,1.929840e+05,192984.000000,192984.000000,192984.000000,192984.000000
mean,52.712150,26.860537,14.470625,939.805608,16.794952,6.075046,52.241040,53.628155,51.232897,943.015291,...,6.173400,6.157256,-1.845999e-17,-5.550655e-17,-0.002785,-3.554140e-03,-0.002997,-0.000684,0.287278,0.414501
std,15.722700,49.673873,26.783313,12.797334,9.158462,3.743378,15.767958,16.392558,15.040098,6.651617,...,3.865857,3.848838,7.071086e-01,7.071086e-01,0.706861,7.073415e-01,0.707344,0.706866,0.452493,0.492637
min,9.500000,0.000000,0.000000,869.000000,-12.000000,0.000000,9.500000,9.500000,9.500000,911.000000,...,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000
25%,40.759259,5.333333,2.333333,937.000000,9.694444,3.500000,40.466667,41.333333,39.750000,939.000000,...,3.500000,3.500000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,-0.781831,-0.900969,0.000000,0.000000
50%,53.604167,11.000000,5.750000,942.000000,16.000000,5.300000,53.000000,54.495833,52.250000,943.000000,...,5.400000,5.400000,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,0.000000,-0.222521,0.000000,0.000000
75%,64.850000,27.000000,14.500000,946.000000,23.750000,7.740000,64.250000,66.000000,63.000000,947.000000,...,7.900000,7.900000,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,0.781831,0.623490,1.000000,1.000000
max,99.000000,1995.000000,639.250000,971.750000,47.500000,32.800000,99.000000,99.000000,86.750000,971.750000,...,32.800000,32.800000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,0.974928,1.000000,1.000000,1.000000


In [6]:
len(df)

192984

In [7]:
df['sensorId'].value_counts()

sensorId
16836a55-7140-43e2-9a63-56fac5cba714    17544
2002                                    17544
23b735ef-a996-4a7f-9998-2aa7e78827b0    17544
2819ecbb-5de3-4092-aa1d-4ba3a8c40add    17544
40f081a6-4095-43f7-bffb-64e2af8c026e    17544
7b316592-8036-41e2-b8dc-b06b6a9afd54    17544
874ff9c6-786d-45fc-a90e-48c7ffe03417    17544
87f82783-853b-417d-8964-b5cf11e44873    17544
d241a044-0a06-40c2-9d90-c91fd0a95060    17544
d7060523-b163-4cc3-bc94-970dac72a38a    17544
fec52a19-9148-4350-a1b4-ae0da05ee199    17544
Name: count, dtype: int64

In [8]:
TARGET = 'pm10'
ID_COL = 'sensorId'
TIME_COL = 'timestamp'
PREDICTION_LENGTH = 512

In [9]:
df[TIME_COL] = pd.to_datetime(df[TIME_COL])

In [10]:
df = df.drop(columns=['pm25','city'],axis=1)

In [11]:
df.columns

Index(['timestamp', 'sensorId', 'humidity', 'pm10', 'pressure', 'temperature',
       'wind_speed', 'neighbor1_humidity', 'neighbor2_humidity',
       'neighbor3_humidity', 'neighbor1_pressure', 'neighbor2_pressure',
       'neighbor3_pressure', 'neighbor1_temperature', 'neighbor2_temperature',
       'neighbor3_temperature', 'neighbor1_wind_speed', 'neighbor2_wind_speed',
       'neighbor3_wind_speed', 'hour_sin', 'hour_cos', 'month_sin',
       'month_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_heating_season'],
      dtype='object')

In [12]:
df.isnull().sum()

timestamp                    0
sensorId                     0
humidity                     0
pm10                     43212
pressure                     0
temperature                  0
wind_speed                   0
neighbor1_humidity           0
neighbor2_humidity           0
neighbor3_humidity           0
neighbor1_pressure           0
neighbor2_pressure           0
neighbor3_pressure           0
neighbor1_temperature        0
neighbor2_temperature        0
neighbor3_temperature        0
neighbor1_wind_speed         0
neighbor2_wind_speed         0
neighbor3_wind_speed         0
hour_sin                     0
hour_cos                     0
month_sin                    0
month_cos                    0
day_sin                      0
day_cos                      0
is_weekend                   0
is_heating_season            0
dtype: int64

In [13]:
def clean_outliers_spatiotemporal(df, target_col='pm10', window_size=4, threshold=2.5):

    df = df.sort_values('timestamp')

    rolling_stats = df.set_index('timestamp')[target_col].rolling(window=f'{window_size}h')

    global_rolling_mean = rolling_stats.mean().reset_index(drop=True)
    global_rolling_std = rolling_stats.std().reset_index(drop=True)


    z_scores = (df[target_col] - global_rolling_mean) / (global_rolling_std + 1e-6)

    df[target_col] = df[target_col].mask(z_scores.abs() > threshold, global_rolling_mean)

    return df

In [14]:
df = clean_outliers_spatiotemporal(df)

In [15]:
df.describe()

,humidity,pm10,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,neighbor1_pressure,neighbor2_pressure,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
count,192984.000000,149772.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,192984.000000,...,192984.000000,192984.000000,1.929840e+05,1.929840e+05,192984.000000,1.929840e+05,192984.000000,192984.000000,192984.000000,192984.000000
mean,52.712150,15.646266,939.805608,16.794952,6.075046,52.241040,53.628155,51.232897,943.015291,942.100966,...,6.173400,6.157256,-1.892483e-17,-5.550425e-17,-0.002785,-3.554140e-03,-0.002997,-0.000684,0.287278,0.414501
std,15.722700,19.536842,12.797334,9.158462,3.743378,15.767958,16.392558,15.040098,6.651617,6.727275,...,3.865857,3.848838,7.071086e-01,7.071086e-01,0.706861,7.073415e-01,0.707344,0.706866,0.452493,0.492637
min,9.500000,0.000000,869.000000,-12.000000,0.000000,9.500000,9.500000,9.500000,911.000000,911.000000,...,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000
25%,40.759259,5.250000,937.000000,9.694444,3.500000,40.466667,41.333333,39.750000,939.000000,938.000000,...,3.500000,3.500000,-7.071068e-01,-7.071068e-01,-0.500000,-8.660254e-01,-0.781831,-0.900969,0.000000,0.000000
50%,53.604167,9.584007,942.000000,16.000000,5.300000,53.000000,54.495833,52.250000,943.000000,942.000000,...,5.400000,5.400000,6.123234e-17,-6.123234e-17,0.000000,-1.836970e-16,0.000000,-0.222521,0.000000,0.000000
75%,64.850000,19.000000,946.000000,23.750000,7.740000,64.250000,66.000000,63.000000,947.000000,946.000000,...,7.900000,7.900000,7.071068e-01,7.071068e-01,0.866025,5.000000e-01,0.781831,0.623490,1.000000,1.000000
max,99.000000,507.500000,971.750000,47.500000,32.800000,99.000000,99.000000,86.750000,971.750000,971.750000,...,32.800000,32.800000,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,0.974928,1.000000,1.000000,1.000000


In [16]:
numeric_features = [
    'humidity', 'pressure', 'temperature', 'wind_speed',
    'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
    'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
    'neighbor1_temperature', 'neighbor2_temperature', 'neighbor3_temperature',
    'neighbor1_wind_speed', 'neighbor2_wind_speed', 'neighbor3_wind_speed'
]
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

In [17]:
scalerpm10 = StandardScaler()
df['pm10'] = scalerpm10.fit_transform(df[['pm10']])
df

,timestamp,sensorId,humidity,pm10,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2023-12-01 00:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.357521,0.649394,0.249615,-0.523556,0.594372,0.723291,0.611113,0.825325,...,0.550100,0.213765,0.000000,1.000000,-0.500000,0.866025,-0.433884,-0.900969,0,1
157896,2023-12-01 00:00:00+00:00,d7060523-b163-4cc3-bc94-970dac72a38a,0.695409,NaN,0.171473,-0.584974,0.241749,0.936012,0.611113,0.649406,...,0.550100,-0.300677,0.000000,1.000000,-0.500000,0.866025,-0.433884,-0.900969,0,1
35088,2023-12-01 00:00:00+00:00,23b735ef-a996-4a7f-9998-2aa7e78827b0,0.695409,NaN,0.171473,-0.584974,0.241749,0.555493,0.815729,0.825325,...,0.550100,0.213765,0.000000,1.000000,-0.500000,0.866025,-0.433884,-0.900969,0,1
17544,2023-12-01 00:00:00+00:00,2002,0.695409,0.581147,0.171473,-0.584974,0.594372,0.723291,0.815729,0.472102,...,0.550100,0.556726,0.000000,1.000000,-0.500000,0.866025,-0.433884,-0.900969,0,1
175440,2023-12-01 00:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.695409,NaN,0.171473,-0.584974,0.241749,0.386373,0.891983,0.825325,...,-0.303530,0.556726,0.000000,1.000000,-0.500000,0.866025,-0.433884,-0.900969,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35087,2025-11-30 23:00:00+00:00,2002,1.193187,0.453925,-0.338617,-1.096797,-0.313901,1.219640,1.059742,0.848873,...,-0.329397,-0.326659,-0.258819,0.965926,-0.866025,0.500000,-0.781831,0.623490,1,1
157895,2025-11-30 23:00:00+00:00,d241a044-0a06-40c2-9d90-c91fd0a95060,1.369861,-0.775269,-3.970025,-1.506256,0.621086,1.300677,2.706836,0.832251,...,0.006881,0.011106,-0.258819,0.965926,-0.866025,0.500000,-0.781831,0.623490,1,1
17543,2025-11-30 23:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,0.717935,1.118595,0.269150,-0.960311,-0.313901,0.729897,1.088550,1.345695,...,-0.329397,-0.326659,-0.258819,0.965926,-0.866025,0.500000,-0.781831,0.623490,1,1
122807,2025-11-30 23:00:00+00:00,874ff9c6-786d-45fc-a90e-48c7ffe03417,2.880419,0.069292,0.093332,-1.178689,0.033380,0.729897,1.166499,0.848873,...,0.006881,-0.326659,-0.258819,0.965926,-0.866025,0.500000,-0.781831,0.623490,1,1


In [18]:
context_df = df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
ground_truth_df = df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)

/tmp/ipykernel_91038/2320177275.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  context_df = df.groupby(ID_COL).apply(lambda x: x.iloc[:-PREDICTION_LENGTH]).reset_index(drop=True)
/tmp/ipykernel_91038/2320177275.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ground_truth_df = df.groupby(ID_COL).apply(lambda x: x.iloc[-PREDICTION_LENGTH:]).reset_index(drop=True)


In [19]:
ground_truth_df

,timestamp,sensorId,humidity,pm10,pressure,temperature,wind_speed,neighbor1_humidity,neighbor2_humidity,neighbor3_humidity,...,neighbor2_wind_speed,neighbor3_wind_speed,hour_sin,hour_cos,month_sin,month_cos,day_sin,day_cos,is_weekend,is_heating_season
0,2025-11-09 16:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,1.179053,1.476893,0.327756,-0.086800,-0.260473,1.015287,1.249870,1.214563,...,-0.277662,-0.274696,-0.866025,-5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
1,2025-11-09 17:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,1.226755,0.854136,0.327756,-0.086800,-0.126903,1.142127,1.322159,1.380786,...,-0.148324,-0.144786,-0.965926,-2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
2,2025-11-09 18:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,1.290357,0.568350,0.347291,-0.114097,-1.222174,1.157982,1.386314,1.497142,...,-1.208894,-1.210045,-1.000000,-1.836970e-16,-0.866025,0.5,-0.781831,0.62349,1,1
3,2025-11-09 19:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,1.338059,-0.263414,0.405897,-0.195989,-0.420756,1.189692,1.420912,1.580253,...,-0.432868,-0.430587,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
4,2025-11-09 20:00:00+00:00,16836a55-7140-43e2-9a63-56fac5cba714,1.306258,-0.314600,0.405897,-0.195989,-0.554326,1.231972,1.281566,1.613498,...,-0.562205,-0.560497,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5627,2025-11-30 19:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.574830,-0.225025,0.171473,-0.851122,-0.046762,0.714042,2.005296,1.153615,...,-0.070722,-0.222732,-0.965926,2.588190e-01,-0.866025,0.5,-0.781831,0.62349,1,1
5628,2025-11-30 20:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.670233,-0.340193,0.171473,-0.851122,-0.100189,0.729897,2.249310,1.203482,...,-0.122457,-0.196750,-0.866025,5.000000e-01,-0.866025,0.5,-0.781831,0.62349,1,1
5629,2025-11-30 21:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.686134,-0.186636,0.210544,-0.933014,-0.073476,0.703472,2.483156,1.276127,...,-0.096589,-0.196750,-0.707107,7.071068e-01,-0.866025,0.5,-0.781831,0.62349,1,1
5630,2025-11-30 22:00:00+00:00,fec52a19-9148-4350-a1b4-ae0da05ee199,0.702035,-0.634509,0.249615,-0.987609,-0.126903,0.729897,2.706836,1.306909,...,-0.148324,-0.300677,-0.500000,8.660254e-01,-0.866025,0.5,-0.781831,0.62349,1,1


In [20]:
ground_truth_df['timestamp'].max()

Timestamp('2025-11-30 23:00:00+0000', tz='UTC')

In [21]:
ground_truth_df['sensorId'].value_counts()

sensorId
16836a55-7140-43e2-9a63-56fac5cba714    512
2002                                    512
23b735ef-a996-4a7f-9998-2aa7e78827b0    512
2819ecbb-5de3-4092-aa1d-4ba3a8c40add    512
40f081a6-4095-43f7-bffb-64e2af8c026e    512
7b316592-8036-41e2-b8dc-b06b6a9afd54    512
874ff9c6-786d-45fc-a90e-48c7ffe03417    512
87f82783-853b-417d-8964-b5cf11e44873    512
d241a044-0a06-40c2-9d90-c91fd0a95060    512
d7060523-b163-4cc3-bc94-970dac72a38a    512
fec52a19-9148-4350-a1b4-ae0da05ee199    512
Name: count, dtype: int64

In [22]:
print("Loading Chronos-2 and generating forecasts...")
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="auto",
    dtype=torch.bfloat16,
)

Loading Chronos-2 and generating forecasts...


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: db24dd84-c36c-44b0-ae1e-65b68499ed6c)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-2/0f8a440441931157957e2be1a9bce66627d99c76/config.json
Retrying in 1s [Retry 1/5].


In [23]:
future_df = ground_truth_df.drop(columns='pm10',axis=1)

In [24]:
forecast_df = pipeline.predict_df(
    df=context_df,
    prediction_length=PREDICTION_LENGTH,
    target=TARGET,
    id_column=ID_COL,
    future_df = future_df
)

In [25]:
# Merge predictions with actual values to align them
eval_df = pd.merge(
    forecast_df[[ID_COL, TIME_COL, 'predictions']],
    ground_truth_df[[ID_COL, TIME_COL, TARGET]],
    on=[ID_COL, TIME_COL]
)

In [26]:
eval_df.columns

Index(['sensorId', 'timestamp', 'predictions', 'pm10'], dtype='object')

In [27]:
eval_df

,sensorId,timestamp,predictions,pm10
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 16:00:00+00:00,0.863281,1.476893
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 17:00:00+00:00,0.941406,0.854136
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 18:00:00+00:00,0.808594,0.568350
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 19:00:00+00:00,0.671875,-0.263414
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 20:00:00+00:00,0.562500,-0.314600
...,...,...,...,...
5627,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 19:00:00+00:00,-0.527344,-0.225025
5628,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 20:00:00+00:00,-0.578125,-0.340193
5629,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 21:00:00+00:00,-0.621094,-0.186636
5630,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 22:00:00+00:00,-0.660156,-0.634509


In [28]:
eval_df['predictions'] = scalerpm10.inverse_transform(eval_df[['predictions']])
eval_df['pm10'] = scalerpm10.inverse_transform(eval_df[['pm10']])

In [29]:
eval_df

,sensorId,timestamp,predictions,pm10
0,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 16:00:00+00:00,32.511997,44.500000
1,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 17:00:00+00:00,34.038311,32.333333
2,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 18:00:00+00:00,31.443583,26.750000
3,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 19:00:00+00:00,28.772539,10.500000
4,16836a55-7140-43e2-9a63-56fac5cba714,2025-11-09 20:00:00+00:00,26.635704,9.500000
...,...,...,...,...
5627,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 19:00:00+00:00,5.343669,11.250000
5628,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 20:00:00+00:00,4.351567,9.000000
5629,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 21:00:00+00:00,3.512097,12.000000
5630,fec52a19-9148-4350-a1b4-ae0da05ee199,2025-11-30 22:00:00+00:00,2.748941,3.250000


In [30]:
eval_df.isnull().sum()

sensorId        0
timestamp       0
predictions     0
pm10           61
dtype: int64

In [31]:
eval_df.dropna(inplace=True)

In [32]:
y_true = eval_df['pm10']
y_pred = eval_df['predictions']

In [33]:
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"--- Global Model Evaluation ---")
print(f"RMSE: {rmse:.4f}")
print(f"R2 score: {r2:.4f}")

--- Global Model Evaluation ---
RMSE: 13.5762
R2 score: 0.4155


In [34]:
for sensor in eval_df['sensorId'].unique():
  sensor_data = eval_df[eval_df['sensorId']==sensor]
  y_true = sensor_data['pm10']
  y_pred = sensor_data['predictions']
  rmse = np.sqrt(mean_squared_error(y_true, y_pred))
  r2 = r2_score(y_true, y_pred)

  print(f"--- {sensor} Model Evaluation ---")
  print(f"RMSE: {rmse:.4f}")
  print(f"R2 score: {r2:.4f}")
  print(f"Count: {len(sensor_data)}")

--- 16836a55-7140-43e2-9a63-56fac5cba714 Model Evaluation ---
RMSE: 19.6715
R2 score: 0.0624
Count: 512
--- 2002 Model Evaluation ---
RMSE: 12.0497
R2 score: -0.0840
Count: 464
--- 23b735ef-a996-4a7f-9998-2aa7e78827b0 Model Evaluation ---
RMSE: 7.0114
R2 score: -0.1567
Count: 512
--- 2819ecbb-5de3-4092-aa1d-4ba3a8c40add Model Evaluation ---
RMSE: 7.4929
R2 score: -0.0312
Count: 512
--- 40f081a6-4095-43f7-bffb-64e2af8c026e Model Evaluation ---
RMSE: 13.2969
R2 score: 0.1417
Count: 512
--- 7b316592-8036-41e2-b8dc-b06b6a9afd54 Model Evaluation ---
RMSE: 30.1883
R2 score: 0.4402
Count: 512
--- 874ff9c6-786d-45fc-a90e-48c7ffe03417 Model Evaluation ---
RMSE: 9.7707
R2 score: 0.0274
Count: 512
--- 87f82783-853b-417d-8964-b5cf11e44873 Model Evaluation ---
RMSE: 9.7586
R2 score: -0.3246
Count: 512
--- d241a044-0a06-40c2-9d90-c91fd0a95060 Model Evaluation ---
RMSE: 3.8841
R2 score: -0.0767
Count: 499
--- d7060523-b163-4cc3-bc94-970dac72a38a Model Evaluation ---
RMSE: 7.9526
R2 score: 0.0752
Coun

In [35]:
import joblib

feature_scaler = scaler
pm10_scaler = scalerpm10

joblib.dump(feature_scaler, "feature_scaler.pkl")
joblib.dump(pm10_scaler, "pm10_scaler.pkl")


['pm10_scaler.pkl']

In [36]:
pipeline.save_pretrained("bitola_chronos_pipeline_pm10")